# README

- **Author**: 楊喬茵（AI協助：Claude Code，Claude Opus 5.5）
- **Created At**: 2026-11-18
- **Last Modified At**: 2026-11-18

---

## What does this file do?

- 從中選會 2026（民國115年）選舉公報抽出 280 位候選人（全國縣市長 81、臺北市議員 98、高雄市議員 101）的公報內容，供策略分析的政見分析使用。觀察單位為候選人，以 `candidate_key` 連結。
- 2026 公報各縣市採同一張候選人卡片模板：標題列（號次‧姓名｜學歷｜經歷）、左欄（號次、姓名、照片、基本資料、推薦之政黨、具結情形）、學歷與經歷欄，以及橫跨兩欄的政見框。各縣市向量框線畫法不一，因此從渲染圖偵測框線切出卡片（[bulletin_2026_helpers.py](bulletin_2026_helpers.py)）；花蓮縣版面有粗外框，4 張卡片改用目視量測的手動座標。
- 文字層與 OCR 都讀不出姓名的 4 張高雄議員卡片，姓名由 AI 目視公報圖像判讀，並以同卡片的號次、出生日期、政黨核對（`match_method = manual_visual_name_reading`，清單在 `bulletin_2026_helpers.MANUAL_NAMES`）。
- 選區公報的所有候選人卡片都已對上他人、仍無卡片的登記者（游智彬），狀態記為 `not_listed_in_bulletin_all_district_cards_matched`：公報未列，可能未通過資格審定或撤回，需另行查證，不代表其他原因。
- 姓名比對依序為：卡片姓名完全相符、姓名出現在左欄文字中；讀不出姓名時（大字直書、姓名為圖片），改用公報出生日期對應 G1 已核對生日（同縣市、職位、選區內唯一），最後才用同選區僅剩一張卡片與一位候選人的排除法。採用方式記在 `match_method`。
- 每一格同時保留 PDF 文字層與 Windows 內建 OCR（zh-Hant-TW，[ocr_windows_20261118.ps1](ocr_windows_20261118.ps1)）結果；文字層完整時採文字層，缺字（例如澎湖縣字型無對應字碼）、掃描檔（南投縣）或政見為設計圖片時採 OCR。每格的採用來源記在 `*_source` 欄。
- 不修改公報原文；不做政見編碼或分類。OCR 文字可能有錯字、斷行或欄位順序錯亂，分析前需對照原 PDF。

---

## What does this file take?

- **Source Data Sets** (one entry per actual file):
  1. `data/raw/cec_bulletin_2026/file_manifest.json`
     - Input key: `manifest`
     - Description: 44 份公報 PDF 的路徑、中選會來源網址、下載時間與 SHA-256（由 [collect_bulletin_2026.py](collect_bulletin_2026.py) 下載）。讀取前逐檔核對 hash。臺東縣縣長公報在 eebulletin.cec.gov.tw 的連結為 404（記於 `download_failures.json`），改自中選會 bulletin.cec.gov.tw 的縣市長公報目錄下載，manifest 該筆 `note` 註明來源。
     - SHA-256: `7d2e3e9304b88c9c1385ee3a72d9c7020907f41192ad98d10c82da0f9a911313`
  2. `data/processed/candidate_social_links_scope_20261114.csv`
     - Input key: `scope`
     - Description: 280 位範圍候選人名冊（僅使用 candidate_key、姓名、縣市、職位）。
     - SHA-256: `d20a9aaafba91da223218505a544c571aa631461b285e364d351ff08c06424e1`
  3. `data/processed/candidates_g1_g6_current.csv`
     - Input key: `g1`
     - Description: G1–G6 主表；只用已核對的 `birth_date`，在卡片姓名讀不出時以出生日期對應候選人，並用 `G2_sex` 核對公報性別。
     - SHA-256: `790407203ac8d7d4de3bc605fe30fe883007902978f89fa416e7e5593456de02`

---

## What does this file output?

- `data/processed/candidate_bulletin_2026.csv`
  - Output key: `bulletin`
  - Description: 280 列，一位候選人一列：公報號次、姓名、出生年月日（民國）、性別、出生地、推薦之政黨、學歷、經歷、政見全文，以及每欄採用來源、政見圖片比例、來源檔案與頁碼。
  - Role: collection / text corpus（政見分析輸入；分析前仍需對照原 PDF 抽查）
  - Expected schema and row-count rules: 280 列、`candidate_key` 唯一且與名冊相同；找不到卡片者 `bulletin_status` 註明原因、內容留空。
- `data/processed/bulletin_2026_cards_ocr.json`
  - Output key: `ocr`
  - Description: 每張範圍內卡片四格的 OCR 原始行（含座標）與文字層原文，供逐筆核對。

Documentation: 本筆記本（Data Artifacts 列登記於 `PROJECT.md`）。All paths are relative to the project root, not the operating-system root.


In [1]:
from pathlib import Path
import hashlib
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual


In [2]:
import json, re, sys, shutil, subprocess, logging
import numpy as np
import pdfplumber, pypdfium2 as pdfium
logging.disable(logging.WARNING)
sys.path.insert(0, str(PROJECT_ROOT / "data/code"))
import bulletin_2026_helpers as H

INPUTS = {"manifest": "data/raw/cec_bulletin_2026/file_manifest.json",
          "scope": "data/processed/candidate_social_links_scope_20261114.csv",
          "g1": "data/processed/candidates_g1_g6_current.csv"}
EXPECTED_INPUT_SHA256 = {"manifest": "7d2e3e9304b88c9c1385ee3a72d9c7020907f41192ad98d10c82da0f9a911313", "scope": "d20a9aaafba91da223218505a544c571aa631461b285e364d351ff08c06424e1",
                         "g1": "790407203ac8d7d4de3bc605fe30fe883007902978f89fa416e7e5593456de02"}
OUTPUTS = {"bulletin": "data/processed/candidate_bulletin_2026.csv",
           "ocr": "data/processed/bulletin_2026_cards_ocr.json"}
OUTPUT_READ_OPTIONS = {"bulletin": {"dtype": str, "encoding": "utf-8-sig", "keep_default_na": False}}

input_paths = {k: project_file(p, Path(p).parts[1]) for k, p in INPUTS.items()}
output_paths = {k: project_file(p, "processed", must_exist=False) for k, p in OUTPUTS.items()}
assert not set(input_paths.values()) & set(output_paths.values())
existing = [str(p) for p in output_paths.values() if p.exists()]
if existing:
    raise FileExistsError(f"Review existing outputs before replacing them: {existing}")
input_fingerprints = {k: verify_input_sha256(p, EXPECTED_INPUT_SHA256[k]) for k, p in input_paths.items()}

manifest = json.loads(input_paths["manifest"].read_text(encoding="utf-8"))
for m in manifest:
    assert sha256_file(project_file(m["path"], "raw")) == m["sha256"], m["path"]
scope = pd.read_csv(input_paths["scope"], dtype=str, encoding="utf-8-sig", keep_default_na=False)
scope = scope[["candidate_key", "candidate_name", "county_city", "office_type"]]
assert len(scope) == 280 and scope["candidate_key"].is_unique
pdfs = [m for m in manifest if "投開票所" not in m["path"]]
print({"pdf_files": len(manifest), "candidate_bulletins": len(pdfs),
       "scope": scope.groupby("office_type").size().to_dict()})


{'pdf_files': 45, 'candidate_bulletins': 43, 'scope': {'直轄市議員': 199, '直轄市長': 23, '縣（市）長': 58}}


In [3]:
SCALE_OCR = 3
CROPS = PROJECT_ROOT / "data/temp/bulletin_2026_crops"  # disposable, git-ignored
if CROPS.exists():
    shutil.rmtree(CROPS)
CROPS.mkdir(parents=True)

rows = []
for m in pdfs:
    rel = m["path"].split("cec_bulletin_2026/")[1]
    doc = pdfium.PdfDocument(project_file(m["path"], "raw"))
    with pdfplumber.open(project_file(m["path"], "raw")) as pp:
        for pi in range(len(doc)):
            page = doc[pi]
            manual = H.MANUAL_CARDS.get((rel, pi + 1))
            gray = np.asarray(page.render(scale=1).to_pil().convert("L"))
            found = manual or H.detect_cards(gray)
            if not found:
                continue
            words = pp.pages[pi].extract_words()
            images = pp.pages[pi].images
            big = page.render(scale=SCALE_OCR).to_pil().convert("RGB")
            for ci, c in enumerate(found):
                cid = f"{len(rows):04d}"
                rec = dict(card_id=cid, file=m["path"], source_url=m["source_url"], source_sha256=m["sha256"],
                           county_city=m["county_city"], folder=m["office_folder"], page=pi + 1,
                           card_bbox=json.dumps([c["x0"], c["top"], c["x1"], c["bottom"]]),
                           geometry="manual" if manual else "detected")
                for k, (x0, t, x1, b) in H.regions(c).items():
                    rec[k + "_layer"] = H.words_to_text(H.words_in(words, (x0, t, x1, b)))
                    rec[k + "_img"] = round(H.image_share(images, (x0, t, x1, b)), 2)
                    big.crop((int(x0 * SCALE_OCR), int(t * SCALE_OCR), int(x1 * SCALE_OCR), int(b * SCALE_OCR))
                             ).save(CROPS / f"{cid}_{k}.png")
                rows.append(rec)
cards = pd.DataFrame(rows)
print("cards detected:", len(cards))
print(cards.groupby(["county_city", "folder"]).size().to_string())


cards detected: 413
county_city  folder
南投縣          01縣長        2
嘉義市          01市長        5
嘉義縣          01縣長        2
基隆市          01市長        3
宜蘭縣          01縣長       23
屏東縣          01縣長        2
彰化縣          01縣長        4
新北市          01市長        3
新竹市          01市長       10
新竹縣          01縣長        5
桃園市          01市長        3
澎湖縣          01縣長        6
臺中市          01市長        3
臺北市          01市長        6
             02市議員      97
臺南市          01市長        4
臺東縣          01縣長        4
花蓮縣          01縣長        4
苗栗縣          01縣長       26
連江縣          01縣長       35
金門縣          01縣長       55
雲林縣          01縣長        4
高雄市          01市長        5
             02市議員     102


In [4]:
ocr_json = PROJECT_ROOT / "data/temp/bulletin_2026_ocr_raw.json"
script = PROJECT_ROOT / "data/code/ocr_windows_20261118.ps1"
subprocess.run(["powershell", "-NoProfile", "-ExecutionPolicy", "Bypass", "-File", str(script), str(CROPS), str(ocr_json)],
               check=True, capture_output=True)
ocr_raw = json.loads(ocr_json.read_text(encoding="utf-8-sig"))
assert len(ocr_raw) == 4 * len(cards), (len(ocr_raw), len(cards))
for k in ["left", "education", "experience", "platform"]:
    cards[k + "_ocr"] = [H.ocr_join(ocr_raw.get(f"{cid}_{k}.png") or []) for cid in cards["card_id"]]
# Pages whose text layer lost glyphs (no 出生年月日 label survives) are read entirely by OCR.
cards["force_ocr"] = (cards["geometry"] == "manual") | ~cards["left_layer"].str.replace(r"\s", "", regex=True).str.contains("出生年月日")
for k in ["left", "education", "experience", "platform"]:
    chosen = [H.choose_text(l, o, s, f) for l, o, s, f in
              zip(cards[k + "_layer"], cards[k + "_ocr"], cards[k + "_img"], cards["force_ocr"])]
    cards[k] = [c[0] for c in chosen]
    cards[k + "_source"] = [c[1] for c in chosen]
print(pd.concat({k: cards[k + "_source"].value_counts() for k in ["left", "education", "experience", "platform"]}, axis=1).fillna(0).astype(int).to_string())


            left  education  experience  platform
text_layer   238        379         371       171
ocr          175         31          42       242
empty          0          3           0         0


In [5]:
parsed = cards["left"].map(H.parse_left).apply(pd.Series)
cards = pd.concat([cards, parsed], axis=1)
cards["kind"] = cards["folder"].map(lambda f: "議員" if "議員" in f else "長")
scope["kind"] = scope["office_type"].map(lambda o: "議員" if "議員" in o else "長")
scope["core"] = scope["candidate_name"].map(H.core_name)
pools = {k: list(zip(g["candidate_key"], g["core"])) for k, g in scope.groupby(["county_city", "kind"])}
res = [H.match_candidate(l, n, pools.get((c, kd), [])) for l, n, c, kd in
       zip(cards["left"], cards["name_bulletin"], cards["county_city"], cards["kind"])]
cards["candidate_key"] = [r[0] for r in res]
cards["match_method"] = [r[1] for r in res]

# Names read by eye where the text layer and OCR both failed (see H.MANUAL_NAMES).
for i, r in cards[cards["candidate_key"].isna()].iterrows():
    bbox = json.loads(r["card_bbox"])
    name = H.manual_name(r["file"].split("cec_bulletin_2026/")[1], int(r["page"]), bbox[0], bbox[1])
    if name:
        pool = pools.get((r["county_city"], r["kind"]), [])
        key = [k for k, c in pool if c == name]
        assert len(key) == 1, (name, key)
        cards.at[i, "candidate_key"], cards.at[i, "match_method"] = key[0], "manual_visual_name_reading"

# Fallback for names the text layer / OCR could not read (large vertical name text, names
# drawn as images). Within the same county, office and (for councils) district:
#  1. birth date printed on the card equals the verified G1 birth date, unique in the pool;
#  2. otherwise, if exactly one card and one candidate remain unmatched, pair them.
g1 = pd.read_csv(input_paths["g1"], dtype=str, encoding="utf-8-sig", keep_default_na=False)
birth = g1.set_index("candidate_key")["birth_date"].to_dict()
def roc_to_ad(s):
    p = s.split("-")
    return f"{int(p[0]) + 1911:04d}-{int(p[1]):02d}-{int(p[2]):02d}" if len(p) == 3 and all(x.isdigit() for x in p) else ""
def district(key_or_file):
    # Files covering several districts (第7及第8、第12、13、14、15) give no single district.
    m = re.search(r"第0?(\d+)選舉區", key_or_file)
    return m.group(1) if m and not re.search(r"及第|\d、\d", key_or_file) else ""
cards["birth_ad"] = cards["birth_roc"].map(roc_to_ad)
cards["district"] = np.where(cards["kind"] == "議員", cards["file"].map(district), "")
scope["district"] = np.where(scope["kind"] == "議員", scope["candidate_key"].map(district), "")
real = cards["birth_ad"].ne("")  # cards without a birth date (e.g. polling-place tables) are not candidates
for _ in range(2):
    taken = set(cards["candidate_key"].dropna())
    for i, r in cards[cards["candidate_key"].isna() & real].iterrows():
        pool = scope[(scope["county_city"] == r["county_city"]) & (scope["kind"] == r["kind"]) & ~scope["candidate_key"].isin(taken)]
        if r["district"]:
            pool = pool[pool["district"] == r["district"]]
        same_birth = pool[pool["candidate_key"].map(birth).eq(r["birth_ad"])]
        if len(same_birth) == 1:
            cards.at[i, "candidate_key"], cards.at[i, "match_method"] = same_birth["candidate_key"].iloc[0], "birth_date_matches_G1"
            taken.add(same_birth["candidate_key"].iloc[0])
    # partial name read: at least half of the candidate's name characters appear in the
    # card's name area, and the best score is unique within the district pool
    taken = set(cards["candidate_key"].dropna())
    for i, r in cards[cards["candidate_key"].isna() & real & cards["district"].ne("")].iterrows():
        head = H.core_name(re.split(r"基本資料|出生年月日", re.sub(r"\s", "", r["left"]))[0])
        pool = scope[(scope["county_city"] == r["county_city"]) & (scope["kind"] == r["kind"]) & (scope["district"] == r["district"]) & ~scope["candidate_key"].isin(taken)]
        scores = {k: sum(ch in head for ch in c) / len(c) for k, c in zip(pool["candidate_key"], pool["core"]) if c}
        best = sorted(scores.items(), key=lambda kv: -kv[1])
        if best and best[0][1] >= 0.5 and (len(best) == 1 or best[1][1] < best[0][1]):
            cards.at[i, "candidate_key"], cards.at[i, "match_method"] = best[0][0], "partial_name_in_district"
            taken.add(best[0][0])
    # elimination within a single-district council file
    for (cty, kd, dist), grp in cards[cards["candidate_key"].isna() & real & cards["district"].ne("")].groupby(["county_city", "kind", "district"]):
        pool = scope[(scope["county_city"] == cty) & (scope["kind"] == kd) & (scope["district"] == dist) & ~scope["candidate_key"].isin(set(cards["candidate_key"].dropna()))]
        if len(grp) == 1 and len(pool) == 1:
            cards.at[grp.index[0], "candidate_key"], cards.at[grp.index[0], "match_method"] = pool["candidate_key"].iloc[0], "last_remaining_in_district"
    # same elimination for a county head: one unmatched candidate card left in the county's
    # head-of-government bulletin and one unmatched head candidate (files that also carry
    # county-council cards, e.g. 金門縣, leave several unmatched cards and are skipped)
    for (cty, kd), grp in cards[cards["candidate_key"].isna() & real & (cards["kind"] == "長")].groupby(["county_city", "kind"]):
        pool = scope[(scope["county_city"] == cty) & (scope["kind"] == kd) & ~scope["candidate_key"].isin(set(cards["candidate_key"].dropna()))]
        if len(grp) == 1 and len(pool) == 1:
            cards.at[grp.index[0], "candidate_key"], cards.at[grp.index[0], "match_method"] = pool["candidate_key"].iloc[0], "last_remaining_in_county"
hit = cards[cards["candidate_key"].notna()].copy()
# 金門縣's three district bulletins each reprint the same county-head page; keep one copy
# (exact name match first, then the first file). Any other duplicate is an error.
dups = hit[hit["candidate_key"].duplicated(keep=False)]
assert set(dups["county_city"]) <= {"金門縣"}, dups[["candidate_key", "county_city", "file", "page"]]
hit["_exact"] = hit["match_method"].ne("exact_name")
hit = hit.sort_values(["_exact", "file", "page", "card_id"]).drop_duplicates("candidate_key").drop(columns="_exact")
print("duplicate 金門縣 reprints dropped:", len(dups) - dups["candidate_key"].nunique())
print(cards["match_method"].value_counts().to_string())

FIELDS = ["ballot_number", "name_bulletin", "birth_roc", "sex", "birthplace", "party_bulletin",
          "education", "experience", "platform", "left_source", "education_source", "experience_source",
          "platform_source", "platform_img", "match_method", "geometry", "file", "source_url", "source_sha256",
          "page", "card_bbox", "card_id"]
bulletin = scope[["candidate_key", "candidate_name", "county_city", "office_type"]].merge(
    hit[["candidate_key", *FIELDS]], on="candidate_key", how="left")
# Counties whose head-of-government bulletin could not be downloaded (broken CEC link).
failures = json.loads((PROJECT_ROOT / "data/raw/cec_bulletin_2026/download_failures.json").read_text(encoding="utf-8"))
failed = {re.sub(r"^\d+", "", f["source_path"].split("/")[1]) for f in failures}
failed -= {m["county_city"] for m in manifest}  # replaced from another CEC source
# A council candidate is "not listed" when every candidate card in the district's bulletin
# is already matched to someone else (likely not qualified or withdrawn; verify separately).
left_cards = cards[cards["candidate_key"].isna() & real]
open_districts = {(c, d) for c, d in zip(left_cards["county_city"], left_cards["district"]) if d}
bulletin["district"] = bulletin["candidate_key"].map(district)
bulletin["bulletin_status"] = np.select(
    [bulletin["card_id"].notna(),
     bulletin["county_city"].isin(failed),
     bulletin["office_type"].eq("直轄市議員") & ~pd.Series(list(zip(bulletin["county_city"], bulletin["district"]))).isin(open_districts).values],
    ["extracted", "bulletin_pdf_unavailable_cec_link_404", "not_listed_in_bulletin_all_district_cards_matched"],
    "card_not_matched")
bulletin = bulletin.drop(columns="district")
# Reasons for candidates absent from the bulletin, recorded only with a readable source.
NOTES = {
    "2026|直轄市議員|臺北市第6選舉區|游智彬": (
        "公報未列：中選會2026-10-17審定，因繳附喪失中國大陸戶籍證明未滿10年，不符參選資格（兩岸條例第21條）；"
        "本人已向臺北高等行政法院聲請假處分，裁定結果未查。來源：公視新聞網 https://news.pts.org.tw/article/831796"
        "（2026-11-18讀全文；楊喬茵同日提供相同資訊）"),
}
bulletin["bulletin_note"] = bulletin["candidate_key"].map(NOTES).fillna("")
assert set(NOTES) <= set(bulletin.loc[bulletin["bulletin_status"] != "extracted", "candidate_key"])
bulletin = bulletin.rename(columns={"platform": "platform_text", "platform_img": "platform_image_share"}).fillna("")
assert len(bulletin) == 280 and bulletin["candidate_key"].is_unique
assert list(bulletin["candidate_key"]) == list(scope["candidate_key"])
print(bulletin.groupby(["office_type", "bulletin_status"]).size().to_string())
print(bulletin.loc[bulletin["bulletin_status"] != "extracted", ["candidate_name", "county_city", "office_type", "bulletin_status"]].to_string())


duplicate 金門縣 reprints dropped: 14
match_method
exact_name                    259
unmatched                     120
birth_date_matches_G1          20
partial_name_in_district        4
manual_visual_name_reading      4
name_in_left_column             3
last_remaining_in_district      2
last_remaining_in_county        1
office_type  bulletin_status                                  
直轄市議員        extracted                                            198
             not_listed_in_bulletin_all_district_cards_matched      1
直轄市長         extracted                                             23
縣（市）長        extracted                                             58
    candidate_name county_city office_type                                    bulletin_status
161            游智彬         臺北市       直轄市議員  not_listed_in_bulletin_all_district_cards_matched


In [6]:
ok = bulletin[bulletin["bulletin_status"] == "extracted"]
summary = pd.DataFrame({
    "has_birth": (ok["birth_roc"].str.count("-") == 2).mean(),
    "has_sex": ok["sex"].isin(["男", "女"]).mean(),
    "has_party": (ok["party_bulletin"] != "").mean(),
    "education_chars>=4": (ok["education"].map(H.cjk_count) >= 4).mean(),
    "experience_chars>=4": (ok["experience"].map(H.cjk_count) >= 4).mean(),
    "platform_chars>=30": (ok["platform_text"].map(H.cjk_count) >= 30).mean(),
}, index=["share_of_extracted"]).T.round(3)
print(summary.to_string())
print("platform source:", ok["platform_source"].value_counts().to_dict())
# Spot check: parsed sex must agree with the existing G2 sex where both exist.
cmp = ok.merge(g1[["candidate_key", "G2_sex"]], on="candidate_key", how="left")
both = cmp[(cmp["sex"] != "") & (cmp["G2_sex"] != "")]
print("sex agreement with G2:", f"{(both['sex'] == both['G2_sex']).mean():.3f}", "n =", len(both))
print(both.loc[both["sex"] != both["G2_sex"], ["candidate_name", "county_city", "sex", "G2_sex"]].to_string())

ocr_out = {cid: {k: {"text_layer": r[k + "_layer"], "ocr_lines": ocr_raw.get(f"{cid}_{k}.png") or []}
                 for k in ["left", "education", "experience", "platform"]}
           for cid, r in cards.set_index("card_id").iterrows() if cid in set(ok["card_id"])}
outputs = {"bulletin": bulletin}


                     share_of_extracted
has_birth                         0.896
has_sex                           0.989
has_party                         0.982
education_chars>=4                0.986
experience_chars>=4               1.000
platform_chars>=30                0.928
platform source: {'ocr': 184, 'text_layer': 95}
sex agreement with G2: 1.000 n = 275
Empty DataFrame
Columns: [candidate_name, county_city, sex, G2_sex]
Index: []


In [7]:
verification = {}
path = output_paths["bulletin"]
with path.open("x", encoding="utf-8-sig", newline="") as stream:
    bulletin.to_csv(stream, index=False)
reread = pd.read_csv(path, **OUTPUT_READ_OPTIONS["bulletin"])
assert list(reread.columns) == list(bulletin.columns) and len(reread) == 280 and reread["candidate_key"].is_unique
pd.testing.assert_frame_equal(bulletin.astype(str).reset_index(drop=True), reread.astype(str), check_dtype=False)
verification["bulletin"] = {"rows": len(reread), "columns": len(reread.columns), "sha256": sha256_file(path)}
with output_paths["ocr"].open("x", encoding="utf-8") as stream:
    json.dump(ocr_out, stream, ensure_ascii=False)
assert json.loads(output_paths["ocr"].read_text(encoding="utf-8")).keys() == ocr_out.keys()
verification["ocr"] = {"cards": len(ocr_out), "sha256": sha256_file(output_paths["ocr"])}
assert all(sha256_file(p) == input_fingerprints[k] for k, p in input_paths.items()), "An input changed."
print(verification)


{'bulletin': {'rows': 280, 'columns': 28, 'sha256': '742190f88cf9a1f1eb70ca2fdb6e570c7caf4949123805a1fd653184ed5d7950'}, 'ocr': {'cards': 279, 'sha256': '4b032748914700f2928e89589481f11261a7b5ca7db500c6d577d1312e8bc563'}}


## 限制

- 臺東縣縣長公報取自中選會另一公報網站 bulletin.cec.gov.tw（eebulletin 連結 404）。
- OCR 結果（`*_source = ocr`）可能有錯字、斷行與閱讀順序錯亂，尤其是多欄設計的政見圖片與小字掃描文字；文字層結果也可能因版面重疊而混入相鄰欄位文字。用於政見分析前，請以 `file`、`page`、`card_bbox` 回到原 PDF 抽查。
- 卡片格線由影像自動偵測；花蓮縣 4 張卡片為手動座標，區域邊界可能略有偏差。
- 本筆記本只抽取公報原文，不判斷政見真偽或做主題分類。
